In [1]:
from IPython.display import Image

> NCCL(NVIDIA Collective Communications Library): Optimized primitives for inter-GPU communication.

- https://docs.nvidia.com/deeplearning/nccl/user-guide/docs/usage/collectives.html
    - AllReduce
    - Broadcast/Reduce
    - AllGather
    - ReduceScatter
    - AlltoAll
    - Gather/Scatter

### GPU cards topo & basics

In [2]:
Image(url='./imgs/gpu-topo.png', width=400)

- 单机多卡 (Intra-node):
    - NCCL 会直接通过 NVLink 进行 P2P (Peer-to-Peer) 传输，带宽高（如 600GB/s - 900GB/s），延迟极低，完全不经过 CPU 和系统内存。
    - 如果没有 NVLink，数据会走 PCIe 总线。虽然慢点，但 NCCL 依然会利用 P2P Direct Access，让 GPU A 直接读写 GPU B 的显存，尽量绕过 CPU。
- 多机多卡 (Inter-node):
    - 必须过网卡: 数据必须离开机器，走网络。
    - GPUDirect RDMA: 这是关键技术。NCCL 会利用 RDMA (InfiniBand 或 RoCE) 技术，让网卡直接读取 GPU 显存并发出去，不需要先把数据考到 CPU 内存再发。这大幅降低了延迟和 CPU 占用。
        - 但网络带宽（比如 100Gbps, 400Gbps IB）远小于机内的 NVLink 速度，所以多机训练时，梯度压缩或大 Batch size（减少通信频率）变得更重要。

- Bucket（桶）机制： PyTorch 会把参数按顺序装进一个个 Bucket（桶）。一旦某个桶里的梯度算好了，NCCL 的 AllReduce 就会立刻触发。

### AllReduce

- Logical Semantics: Reduce (to Root) + Broadcast
- 实际工程实现（如 Ring-AllReduce 或 Tree-AllReduce）并不会真的选一个“主节点”来做中转，否则那个节点会成为瓶颈。实际算法是大家像接力赛一样通过切片轮转数据（Scatter-Reduce + All-Gather），在传输过程中顺便就完成了计算。
- 在 DDP（分布式数据并行）训练中，AllReduce 的具体物理含义是 “梯度平均”。
    - 输入：GPU $i$ 上根据该 GPU 处理的一小批数据（Mini-batch）计算出来的模型梯度。记为 $g^{(i)}$ (Gradient on GPU $i$).
    - 算子 ($\oplus$): 对应的是向量加法 ($+$)。
    - 计算
        - Sum (求和): NCCL 执行 AllReduce (Op=Sum)：$g_{sum} = \sum_{i=0}^{P-1} g^{(i)}$
            - 此时，每张显卡上都得到了所有卡梯度的总和。
        - Average (求平均): 虽然 NCCL 的 AllReduce 也有 Avg 模式，但 PyTorch 实际上通常是先做 Sum，然后在本地除以卡数 $P$（这是一个非常快的本地操作，不需要通信）。
            - $g_{avg} = \frac{1}{P} \cdot g_{sum}$

### DDP

- 2gpus, dp=2, layers=2：假设总 Batch Size = 32。
    - DDP 会通过 DistributedSampler 把数据切分。
    - GPU 0 拿到前 16 条数据（Data_0）。
    - GPU 1 拿到后 16 条数据（Data_1）。
- forward：完全并行，没有任何通信。
    - GPU 0 计算：$Loss_0 = Model(Data_0)$
    - GPU 1 计算：$Loss_1 = Model(Data_1)$
- backward：梯度是从后往前算的（Layer 2 -> Layer 1）。
    - 时间点 T1：计算 Layer 2 梯度
        - GPU 0: 正在算 Layer 2 在 Data_0 上的梯度 $g2_{gpu0}$。
        - GPU 1: 正在算 Layer 2 在 Data_1 上的梯度 $g2_{gpu1}$。
        - NCCL: 休息中。
    - 时间点 T2：Bucket A (Layer 2) 准备完毕 —— 触发通信
        - 两张卡的 Layer 2 梯度都算好了，被丢进了 Bucket A。
        - PyTorch DDP 引擎检测到 Bucket A 满了，立刻发出指令：“NCCL，把 Bucket A 拿去做 AllReduce！”
    - 时间点 T3：Overlap (重叠) 发生
        - 流水线 1 (计算流 - CUDA Cores):GPU 0 & 1: 既然 Layer 2 算完了，核心立刻转头去算 Layer 1 的梯度 ($g1_{gpu0}$ 和 $g1_{gpu1}$)。
            - 注意：显卡核心在满载工作，没有停下来等数据传输。
        - 流水线 2 (通信流 - NCCL/DMA):
            - NCCL: 正在后台通过 NVLink 或 PCIe搬运 Bucket A (Layer 2) 的数据。它执行 AllReduce (Sum/Avg)：它把 $g2_{gpu0}$ 和 $g2_{gpu1}$ 加起来，求平均。然后把这个平均后的梯度 $g2_{avg}$ 写回两张卡的显存里。
    - 时间点 T4：Layer 1 计算完毕，Layer 2 通信完毕
        - 计算流: Layer 1 的梯度算好了，丢进 Bucket B。
        - 通信流: Bucket A (Layer 2) 的同步刚好完成。现在两张卡上的 Layer 2 梯度都已经变成了相同的 $g2_{avg}$。
    - 时间点 T5：Bucket B (Layer 1) 通信
        - 计算流: 反向传播计算任务全部结束，GPU 计算核心开始空闲（或者开始做一些非梯度的处理）。
        - 通信流: NCCL 开始对 Bucket B (Layer 1) 做 AllReduce。注：因为后面没有更靠前的层需要计算了，所以这里最后的一步通信无法被 Overlap，必须等待完成。
- 参数更新 (Optimizer Step)
    - GPU 0 的显存里有：$g1_{avg}, g2_{avg}$。GPU 1 的显存里有：$g1_{avg}, g2_{avg}$
    - `optimizer.step()`:
        - GPU 0: $W_{new} = W_{old} - lr * g_{avg}$
        - GPU 1: $W_{new} = W_{old} - lr * g_{avg}$

| 时间轴 | GPU 计算核心 (Compute Stream) | NCCL 通信通道 (Comm Stream) | 状态备注 |
| :--- | :--- | :--- | :--- |
| **T1** | 正在算 Layer 2 梯度 | (空闲) | 刚开始 Backward |
| **T2** | Layer 2 算完 -> **提交通信请求** | (准备启动) | Bucket A 满了 |
| **T3** | **正在算 Layer 1 梯度** | **正在传输 Layer 2 梯度** | **<-- Overlap (双轨并行)** |
| **T4** | Layer 1 算完 -> 提交通信请求 | Layer 2 传输完成 | Layer 2 梯度已同步 |
| **T5** | (计算结束，等待) | **正在传输 Layer 1 梯度** | 最后的通信无法掩盖 |
| **T6** | **Optimizer 更新参数** | (空闲) | 全局同步完成 |